In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
import random
import time


from numpy import linalg as LA
import scipy.optimize as opt
from scipy.optimize import minimize
from numpy.random import rand

import networkx as nx

In [2]:
#auxiliary functions
def conv(i,j):
    return i*N+j

def rev_conv(k):
    return [k//N,k-k//N*N]

#Laplacian
def build_L_1(N):
    G = nx.grid_graph(dim=(N,N),periodic=False)
    G=nx.convert_node_labels_to_integers(G)
    L=nx.laplacian_matrix(G).toarray()
    L=to_fixed(L)
    return L

#First derivative in relation to the x axis
def build_L_2(N):
    L=np.zeros((N**2,N**2))
    for i in range (1,N-1):
        for j in range (N):
            k_1,k_2,k_3=conv(i+1,j),conv(i-1,j),conv(i,j)
            L[k_3][k_1]=1
            L[k_3][k_2]=-1
    for j in range(N):
        k_1,k_2,k_3,k_4=conv(0,j),conv(1,j),conv(N-1,j),conv(N-2,j)
        L[k_1][k_2]=1
        
        L[k_3][k_4]=-1
        
    return L

#First derivative in relation to the y axis
def build_L_3(N):
    L=np.zeros((N**2,N**2))
    for i in range (N):
        for j in range (1,N-1):
            k_1,k_2,k_3=conv(i,j+1),conv(i,j-1),conv(i,j)
            L[k_3][k_1]=1
            L[k_3][k_2]=-1
    for i in range(N):
        k_1,k_2,k_3,k_4=conv(i,0),conv(i,1),conv(i,N-1),conv(i,N-2)
        L[k_1][k_2]=1
        
        L[k_3][k_4]=-1
        
    return L

def to_fixed(L):
    L_max=np.max(L)
    for i in range (len(L)):
        L[i][i]=L_max
    return L

In [3]:
#Auxiliary matrix for the Jacobian
def M_1(beta):
    M=np.zeros((N**2,N**2))
    for i in range (N):
        for j in range (N):
            k_1,k_2,k_3,k_4,k_5=conv(i,(j+1)%N),conv(i,(j-1)%N),conv((i+1)%N,j),conv((i-1)%N,j),conv(i,j)
            M[k_5][k_1]=beta[i][j]*L_1[k_5][k_1]
            M[k_5][k_2]=beta[i][j]*L_1[k_5][k_2]
            M[k_5][k_3]=beta[i][j]*L_1[k_5][k_3]
            M[k_5][k_4]=beta[i][j]*L_1[k_5][k_4]
            M[k_5][k_5]=beta[i][j]*L_1[k_5][k_5]
    return M

#Auxiliary matrix for the Jacobian
def M_2(beta):
    M=np.zeros((N**2,N**2))
    for i in range (1,N-1):
        for j in range (N):
            k_1,k_2,k_3=conv(i+1,j),conv(i-1,j),conv(i,j)
            M[k_3][k_1]=(beta[i+1][j]-beta[i-1][j])*L_2[k_3][k_1]
            M[k_3][k_2]=(beta[i+1][j]-beta[i-1][j])*L_2[k_3][k_2]
            
    for j in range (N):
        k_1,k_2,k_3,k_4=conv(0,j),conv(1,j),conv(N-1,j),conv(N-2,j)
        M[k_1][k_2]=-3*beta[0][j]+4*beta[1][j]-beta[2][j]
        M[k_3][k_4]=-3*beta[N-1][j]+4*beta[N-2][j]-beta[N-3][j]
        
    return M

#Auxiliary matrix for the Jacobian
def M_3(beta):
    M=np.zeros((N**2,N**2))
    for i in range (N):
        for j in range (1,N-1):
            k_1,k_2,k_3=conv(i,j+1),conv(i,j-1),conv(i,j)
            M[k_3][k_1]=(beta[i][j+1]-beta[i][j-1])*L_2[k_3][k_1]
            M[k_3][k_2]=(beta[i][j+1]-beta[i][j-1])*L_2[k_3][k_2]
            
    for i in range (N):
        k_1,k_2,k_3,k_4=conv(i,0),conv(i,1),conv(i,N-1),conv(i,N-2)
        M[k_1][k_2]=-3*beta[i][0]+4*beta[i][1]-beta[i][2]

        M[k_3][k_4]=-3*beta[i][N-1]+4*beta[i][N-2]-beta[i][N-3]
        
    return M

#Calculate the optimal homogeneous damping
def calc_hom_damp():
    n_iter=10**(7)
    p0=[1]
    result_min = minimize(objective_hom,x0=p0, method='Nelder-Mead', options={'maxiter':n_iter})
    min_hom_fun=result_min.fun
    min_hom_x=result_min.x
    return min_hom_fun, min_hom_x

In [4]:
#Jacobian
def Jac(x):
    M_beta_1=M_1(x.reshape(N,N))
    M_beta_2=M_2(x.reshape(N,N))
    M_beta_3=M_3(x.reshape(N,N))
    return np.block([[np.zeros((N**2,N**2)),np.identity(N**2)],[-L_1/dx**2,-2*M_beta_1/dx**2-(M_beta_2+M_beta_3)/(2*dx**2)]])

In [5]:
#Kronecker delta
def kron_del(x,y):
    if x==y:
        return 1
    return 0

#Minimization function, maximum Lyapunov exponent
def objective(x):
    J=Jac(x)
    EV=np.real(np.sort(LA.eigvals(J)))
    return EV[len(EV)-1]

In [7]:
#Calculate optimal heterogeneous one
N_min=3#starting number of position modes
N_max=6#ending number of position modes
for N in range (N_min,N_max):
    start=time.time()

    n_p0=10**(2)#number
    eps=0.5
    n_iter=10**(6)

    dx=np.pi/N
    L_1=build_L_1(N)
    L_2=build_L_2(N)
    L_3=build_L_3(N)

    bounds_min=[[0,10] for i in range (N**2)]

    min_fun=0
    min_x=np.zeros(N**2)
    #minimize the maximum Lyapunov exponent for n_p0 different initial conditions
    for i in range (n_p0):
        p0 = 1+ 2*rand(N**2)*eps-eps
        result_min = minimize(objective, x0=p0, method='SLSQP', bounds=bounds_min, options={'maxiter':n_iter})

        if result_min.fun<min_fun:
            min_fun=result_min.fun
            min_x=result_min.x

    #Print solution
    print(N)
    print(min_fun)
    print(np.array(repr(min_x)))

    #Computation time
    end=time.time()
    print()
    print(end-start)
    print()

3
-1.2075510812738732
array([0.29097697, 0.536358  , 0.40371765, 0.55910576, 0.84036341,
       0.53719094, 0.38558491, 0.49413009, 0.37220214])

55.58882474899292

4
-1.0090025114690888
array([0.3463273 , 0.44502115, 0.17667427, 0.34802753, 0.55279849,
       0.59605215, 0.28403463, 0.3683729 , 0.39673492, 0.57607641,
       0.56553964, 0.54874415, 0.158408  , 0.18724171, 0.56489852,
       0.43088917])

115.27478337287903

5
-1.0932643863930471
array([0.52688694, 0.30857853, 0.33284015, 0.18857422, 0.17396783,
       0.42215313, 0.40505154, 0.13376555, 0.50989516, 0.38760467,
       0.49367578, 0.54339082, 0.14076063, 0.53968176, 0.47391689,
       0.38663656, 0.53324227, 0.49384635, 0.53609619, 0.3531523 ,
       0.1779038 , 0.1484345 , 0.45188042, 0.1203133 , 0.09266773])

340.81767988204956

